<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "332"
iso_country = "Haiti"
import_portfolio = "h4_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
year = 2017
gdp = 15.090*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Haiti': 

Number of trade records present in dataset: 18685
Number of partner countries present in dataset: 118
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1669.5m
Total weight within the dataset: 2489.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,332,Haiti,056,Belgium,Import,Sea,960400,960400 | Hand sieves and hand,9604,9604 | Hand sieves and hand,96,96 | Miscellaneous manufa,38999,"Articles n.e.c. (including candles, tapers, sk...",Other Manufacturing,42,OMF - Manufactures n.e.c.,5425.309770,NaN,5425.309770,NaN,0.0,408.663016,610.347349,5855.0,3638.127205,2220.521295,['Container'],['Container'],10.817501,10.797714,10.799694,10.799694,10.808540,10.808540,11.321960,20.144545,12.431538,11.603075,11.635601,11.915358,11.083337,19.695753,13.119421,12.038857,11.756308,12.297614,10.874053,21.520889,16.294823,14.258578,13.181824,12.821792,10.668646,22.850260,18.793340,17.543894,14.105488,13.747076,10.467116,23.229865,22.268585,22.505275,15.233870,14.443075,10.261546,22.746869,21.685752,23.704979,17.747177,14.710043,10.067885,20.209249,21.130133,22.117971,16.406213,14.884029,9.870412,18.571834,18.852698,23.115559,19.183815,14.985059,38999
1,1,332,Haiti,070,Bosnia Herzegovina,Import,Sea,283630,283630 | Carbonates; sodium h,2836,2836 | Carbonates; peroxoca,28,28 | Inorganic chemicals;,3424,Salts and peroxysalts of inorganic acids and m...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",8587.831893,NaN,8587.831893,kg,19154.0,19154.000000,19729.684992,9081.0,5642.670051,111328.102619,['Container'],['Container'],542.346464,541.354428,541.453680,541.453680,541.897203,541.897203,567.638039,1009.967327,623.267844,581.732035,583.362744,597.388660,555.674415,987.466703,657.755553,603.580392,589.414505,616.553424,545.181729,1078.971730,816.957578,714.868352,660.884216,642.833615,534.883437,1145.621142,942.223297,879.581017,707.193027,689.223712,524.779535,1164.652993,1116.458239,1128.324925,763.765628,724.118328,514.473099,1140.437514,1087.237345,1188.473320,889.772837,737.503021,504.763700,1013.211325,1059.380806,1108.907049,822.542245,746.226001,494.863200,931.117870,945.199280,1158.922174,961.800138,751.291223,3424


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $627.0m (37.6%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               161904119.2  364178823.0      9.7     14.6
15 | Animal, vegetable or   92213474.0   93063453.8      5.5      3.7
17 | Sugars and sugar con   81845377.2  155248903.0      4.9      6.2
04 | Dairy produce; birds   54576554.6   27425498.8      3.3      1.1
02 | Meat and edible meat   38981657.2   20402611.7      2.3      0.8
21 | Miscellaneous edible   37521268.9   10694178.3      2.2      0.4
19 | Preparations of cere   32692908.1   12608667.1      2.0      0.5
07 | Vegetables and certa   30677188.2   41619028.6      1.8      1.7
11 | Products of the mill   20297077.4   44055465.8      1.2      1.8
16 | Meat, fish, crustace   20267305.9    4528002.0      1.2      0.2
22 | Beverages, spirits a   19389131.2   54348119.3      1.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $536.6m (32.1%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       362794442.2  448441240.0     21.7     18.0
Food & Beverages  165485742.3  151362157.3      9.9      6.1
Fishing             8347710.8    1104241.2      0.5      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $511.3m (30.6%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
OFD - Food products n.e.c.     135614647.5   82675687.8      8.1      3.3
PDR - Paddy rice               113133410.5  196512734.0      6.8      7.9
VOL - Vegetable oils and fats   93404046.9   96777301.2      5.6      3.9
MIL - Dairy products            53001497.0   26092297.3      3.2      1.0
OMT - Meat products n.e.c.      46758229.5   21801009.2      2.8      0.9
V_F - Vegetables, fruit, nuts   29560576.4   41530894.2      1.8      1.7
SGR - Sugar                     19311191.0   42476835.4      1.2      1.7
OCR - Crops n.e.c.               8701466.3    1991062.5      0.5      0.1
PCR - Processed rice             5775250.2   10031609.6      0.3      0.4
GRO - Cereal grains n.e.c.       3775404.4   15333362.5      0.2      0.6
CMT - Bovine meat prods          1884202.2    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 37.6% 
GTAP 30.6% 
EORA 32.1%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_23078/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,332,Haiti,056,Belgium,Import,Sea,960400,960400 | Hand sieves and hand,9604,9604 | Hand sieves and hand,96,96 | Miscellaneous manufa,38999,"Articles n.e.c. (including candles, tapers, sk...",Other Manufacturing,42,OMF - Manufactures n.e.c.,5425.309770,NaN,5425.309770,NaN,0.0,408.663016,610.347349,5855.0,3638.127205,2220.521295,['Container'],['Container'],10.817501,10.797714,10.799694,10.799694,10.808540,10.808540,11.321960,20.144545,12.431538,11.603075,11.635601,11.915358,11.083337,19.695753,13.119421,12.038857,11.756308,12.297614,10.874053,21.520889,16.294823,14.258578,13.181824,12.821792,10.668646,22.850260,18.793340,17.543894,14.105488,13.747076,10.467116,23.229865,22.268585,22.505275,15.233870,14.443075,10.261546,22.746869,21.685752,23.704979,17.747177,14.710043,10.067885,20.209249,21.130133,22.117971,16.406213,14.884029,9.870412,18.571834,18.852698,23.115559,19.183815,14.985059,38999,NaN
1,1,332,Haiti,070,Bosnia Herzegovina,Import,Sea,283630,283630 | Carbonates; sodium h,2836,2836 | Carbonates; peroxoca,28,28 | Inorganic chemicals;,3424,Salts and peroxysalts of inorganic acids and m...,"Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",8587.831893,NaN,8587.831893,kg,19154.0,19154.000000,19729.684992,9081.0,5642.670051,111328.102619,['Container'],['Container'],542.346464,541.354428,541.453680,541.453680,541.897203,541.897203,567.638039,1009.967327,623.267844,581.732035,583.362744,597.388660,555.674415,987.466703,657.755553,603.580392,589.414505,616.553424,545.181729,1078.971730,816.957578,714.868352,660.884216,642.833615,534.883437,1145.621142,942.223297,879.581017,707.193027,689.223712,524.779535,1164.652993,1116.458239,1128.324925,763.765628,724.118328,514.473099,1140.437514,1087.237345,1188.473320,889.772837,737.503021,504.763700,1013.211325,1059.380806,1108.907049,822.542245,746.226001,494.863200,931.117870,945.199280,1158.922174,961.800138,751.291223,3424,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $482.8m of $1669.5m (28.9%)
Volume: 683.4 of 2489.2 million kilograms (27.5%)
Value-to-Weight: 0.71 from 0.67 US$ per kg (+5.3%)
Average distance to market (volume-weighted): 2976.7 from 3397.0 nm (+-12.4%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    516  202295657.6  259742409.4     12.1     10.4
1      Sugar and Honey    127   81849597.1  155250112.9      4.9      6.2
2                Dairy    158   53164252.2   26641074.7      3.2      1.1
3  Unprocessed Cereals     43   42995458.5  157634479.4      2.6      6.3
4        Meat and Prep    106   36467725.1   19387252.0      2.2      0.8
5           Vegetables    333   34836902.2   44125253.0      2.1      1.8
6        Fats and Oils    125   29800577.0   19540099.7      1.8      0.8
7               Fruits     96    1427141.6    1081262.2      0.1      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,45981464.1,81739396.7,50473370.3,47115103.3,47246784.7,48385390.8,42516811.0,94259264.3,90357767.1,91336691.2,61842325.5,58628262.8,40098388.0,75378687.9,76517072.2,93777626.2,77831845.3,60824086.3
1,Fats and Oils,71181027.0,125396409.9,77903913.0,72756570.2,72960245.7,74801143.9,65923588.1,144399248.9,138651336.6,140487780.4,95215352.9,90312038.6,62252522.5,115830041.4,117559567.2,143251240.8,119156511.7,93633912.4
2,Fruits,716476.6,1273132.2,786366.8,734081.3,736130.5,753890.8,662541.6,1468081.4,1407347.2,1422736.5,963426.9,913329.6,624893.7,1174200.0,1191915.8,1460502.9,1212176.9,947506.4
3,Meat and Prep,18862451.7,33560929.0,20711014.4,19330791.2,19384979.2,19851056.5,17438275.7,38701090.0,37099591.9,37493918.6,25379716.1,24062247.5,16444164.4,30940783.8,31408705.1,38510656.5,31960346.9,24965195.1
4,Processed Cereals,341284410.7,587449381.7,370791674.5,347579330.2,348528759.2,357567841.7,317423096.0,684668597.4,654946682.5,665509530.1,455393525.6,427561393.5,300767301.2,549914831.6,557430041.7,673173535.9,561648399.0,442381130.0
5,Sugar and Honey,171117729.5,285893884.7,184190498.8,173321029.6,173782280.1,178580359.4,159987974.0,336367792.3,321166080.0,327983371.3,226614256.6,210962815.6,152219140.7,271349924.6,274706223.7,327143112.2,274263340.6,217730777.4
6,Unprocessed Cereals,240021208.0,426100341.5,263353363.1,245875396.7,246563441.1,252523489.8,221991015.3,491712818.4,471297790.7,476481070.5,322768455.4,305810324.8,209405479.0,393237158.2,399145350.7,488899894.2,405886625.2,317226785.8
7,Vegetables,103744452.6,184166402.4,113825250.9,106271342.2,106567331.9,109145748.6,95953174.3,212388252.2,203587699.0,205861392.1,139436330.4,132175923.2,90512661.2,169908344.7,172474425.1,211221380.6,175311539.5,137111055.2


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.67,0.08,0.02,0.02,0.05,0.97,0.90,0.92,0.36,0.30,0.66,0.69,1.01,0.71,0.39
1,Fats and Oils,1.82,0.23,0.05,0.06,0.12,2.63,2.44,2.50,0.98,0.82,1.80,1.86,2.72,1.91,1.05
2,Fruits,0.39,0.05,0.01,0.01,0.03,0.56,0.52,0.53,0.21,0.18,0.38,0.40,0.59,0.41,0.23
3,Meat and Prep,0.40,0.05,0.01,0.01,0.03,0.58,0.54,0.55,0.22,0.18,0.40,0.41,0.61,0.43,0.23
4,Processed Cereals,1.22,0.15,0.03,0.04,0.08,1.82,1.67,1.72,0.68,0.54,1.23,1.27,1.84,1.29,0.70
5,Sugar and Honey,1.40,0.16,0.03,0.03,0.09,2.15,1.97,2.05,0.81,0.62,1.46,1.50,2.14,1.49,0.80
6,Unprocessed Cereals,4.33,0.54,0.14,0.15,0.29,6.27,5.80,5.92,2.34,1.95,4.28,4.41,6.50,4.57,2.51
7,Vegetables,2.31,0.29,0.07,0.08,0.16,3.34,3.09,3.15,1.25,1.04,2.28,2.35,3.46,2.43,1.34
